# 3.5 Format Data
## CRISP-DM Phase 3: Data Preparation

**Purpose:** Apply final formatting transformations to the feature-engineered datasets (3.3 output) so they are ready for the modeling pipeline (Phase 4).

**Input:** `data/processed/{train,test}_features.csv` from task 3.3
**Output:** `data/processed/{train,test}_formatted.csv` — modeling-ready datasets

**Formatting operations:**
1. Drop raw/intermediate columns (Name, Ticket, Cabin)
2. One-hot encode remaining string columns (Title, Deck)
3. Ensure all features are numeric
4. Reorder columns (ID → Target → Features)
5. Validate final schema

In [1]:
# Setup & path resolution
from pathlib import Path

PROJECT_ROOT = Path(__file__).resolve().parent.parent if "__file__" in dir() else Path.cwd()
if (PROJECT_ROOT / "notebooks").is_dir():
    pass  # cwd is project root
elif (PROJECT_ROOT.parent / "notebooks").is_dir():
    PROJECT_ROOT = PROJECT_ROOT.parent  # cwd is a subdirectory

PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

import pandas as pd
import numpy as np

print(f"Project root: {PROJECT_ROOT}")
print(f"Processed dir: {PROCESSED_DIR}")

Project root: /Users/tba8ydd/Documents/claude-template
Processed dir: /Users/tba8ydd/Documents/claude-template/data/processed


## 1. Load Feature-Engineered Data (3.3 Output)

In [2]:
train = pd.read_csv(PROCESSED_DIR / "train_features.csv")
test = pd.read_csv(PROCESSED_DIR / "test_features.csv")

print(f"Train shape: {train.shape}")
print(f"Test shape:  {test.shape}")
print(f"\nTrain columns: {list(train.columns)}")
print(f"\nTrain dtypes:\n{train.dtypes}")
print(f"\nString columns: {list(train.select_dtypes(include='object').columns)}")

Train shape: (891, 22)
Test shape:  (418, 21)

Train columns: ['PassengerId', 'Survived', 'Pclass', 'Name', 'Sex', 'Age', 'SibSp', 'Parch', 'Ticket', 'Fare', 'Cabin', 'AgeMissing', 'Title', 'FamilySize', 'FamilySizeBin', 'IsAlone', 'HasCabin', 'Deck', 'FareLog', 'TicketGroupSize', 'Embarked_Q', 'Embarked_S']

Train dtypes:
PassengerId          int64
Survived             int64
Pclass               int64
Name                   str
Sex                  int64
Age                float64
SibSp                int64
Parch                int64
Ticket                 str
Fare               float64
Cabin                  str
AgeMissing           int64
Title                  str
FamilySize           int64
FamilySizeBin        int64
IsAlone              int64
HasCabin             int64
Deck                   str
FareLog            float64
TicketGroupSize      int64
Embarked_Q           int64
Embarked_S           int64
dtype: object

String columns: ['Name', 'Ticket', 'Cabin', 'Title', 'Deck']


/var/folders/c1/0_yrgfd54sl_3grrp8bnb1080000gp/T/ipykernel_24282/3419643095.py:8: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  print(f"\nString columns: {list(train.select_dtypes(include='object').columns)}")


## 2. Drop Raw/Intermediate Columns

These columns were engineering sources — their signals have been extracted into derived features:
- **Name** → Title (extracted in 3.3)
- **Ticket** → TicketGroupSize (computed in 3.3)
- **Cabin** → HasCabin + Deck (derived in 3.3)

In [3]:
cols_to_drop = ["Name", "Ticket", "Cabin"]

train = train.drop(columns=cols_to_drop)
test = test.drop(columns=cols_to_drop)

print(f"Dropped: {cols_to_drop}")
print(f"Train shape after drop: {train.shape}")
print(f"Test shape after drop:  {test.shape}")

Dropped: ['Name', 'Ticket', 'Cabin']
Train shape after drop: (891, 19)
Test shape after drop:  (418, 18)


## 3. Encode Remaining String Columns

Two string columns remain after 3.3:
- **Title** (Mr, Mrs, Miss, Master) → one-hot, drop "Mr" (most frequent, male reference)
- **Deck** (A-G, Unknown) → one-hot, drop "Unknown" (77% of passengers, reference category)

In [4]:
def encode_remaining_strings(df: pd.DataFrame) -> pd.DataFrame:
    """One-hot encode Title and Deck, dropping reference categories."""
    out = df.copy()

    # Title: one-hot with Mr as reference (dropped)
    title_dummies = pd.get_dummies(out["Title"], prefix="Title", drop_first=False)
    title_dummies = title_dummies.astype(int)
    if "Title_Mr" in title_dummies.columns:
        title_dummies = title_dummies.drop(columns=["Title_Mr"])
    out = pd.concat([out, title_dummies], axis=1)
    out = out.drop(columns=["Title"])

    # Deck: one-hot with Unknown as reference (dropped)
    deck_dummies = pd.get_dummies(out["Deck"], prefix="Deck", drop_first=False)
    deck_dummies = deck_dummies.astype(int)
    if "Deck_Unknown" in deck_dummies.columns:
        deck_dummies = deck_dummies.drop(columns=["Deck_Unknown"])
    out = pd.concat([out, deck_dummies], axis=1)
    out = out.drop(columns=["Deck"])

    return out

train = encode_remaining_strings(train)
test = encode_remaining_strings(test)

# Align columns: test may be missing one-hot columns that only appear in train
# (e.g., Deck_T has only 1 observation in train, 0 in test)
for col in train.columns:
    if col not in test.columns and col != "Survived":
        test[col] = 0
        print(f"Added missing column to test with zeros: {col}")

# Drop any test columns not in train (shouldn't happen, but safety check)
extra_test_cols = [c for c in test.columns if c not in train.columns]
if extra_test_cols:
    test = test.drop(columns=extra_test_cols)
    print(f"Dropped extra test columns: {extra_test_cols}")

print(f"Train shape after encoding: {train.shape}")
print(f"Test shape after encoding:  {test.shape}")
print(f"\nRemaining string columns: {list(train.select_dtypes(include='object').columns)}")
print(f"\nAll columns:\n{list(train.columns)}")

Added missing column to test with zeros: Deck_T
Train shape after encoding: (891, 28)
Test shape after encoding:  (418, 27)

Remaining string columns: []

All columns:
['PassengerId', 'Survived', 'Pclass', 'Sex', 'Age', 'SibSp', 'Parch', 'Fare', 'AgeMissing', 'FamilySize', 'FamilySizeBin', 'IsAlone', 'HasCabin', 'FareLog', 'TicketGroupSize', 'Embarked_Q', 'Embarked_S', 'Title_Master', 'Title_Miss', 'Title_Mrs', 'Deck_A', 'Deck_B', 'Deck_C', 'Deck_D', 'Deck_E', 'Deck_F', 'Deck_G', 'Deck_T']


## 4. Column Ordering

Reorder columns: ID → Target → Numeric features → One-hot features.

In [5]:
# Define column ordering
id_cols = ["PassengerId"]
target_cols = ["Survived"]
numeric_features = [
    "Pclass", "Sex", "Age", "Fare", "FareLog",
    "SibSp", "Parch", "FamilySize", "FamilySizeBin", "IsAlone",
    "HasCabin", "TicketGroupSize", "AgeMissing",
]
onehot_features = sorted([c for c in train.columns if c.startswith(("Title_", "Deck_", "Embarked_"))])

# Train: ID + Target + Features
train_col_order = id_cols + target_cols + numeric_features + onehot_features
# Test: ID + Features (no target)
test_col_order = id_cols + numeric_features + onehot_features

# Verify all columns accounted for
train_missing = set(train.columns) - set(train_col_order)
train_extra = set(train_col_order) - set(train.columns)
test_missing = set(test.columns) - set(test_col_order)
test_extra = set(test_col_order) - set(test.columns)

print(f"Train columns not in order: {train_missing}")
print(f"Order columns not in train: {train_extra}")
print(f"Test columns not in order:  {test_missing}")
print(f"Order columns not in test:  {test_extra}")

assert not train_missing, f"Unaccounted train columns: {train_missing}"
assert not train_extra, f"Missing train columns: {train_extra}"
assert not test_missing, f"Unaccounted test columns: {test_missing}"
assert not test_extra, f"Missing test columns: {test_extra}"

train = train[train_col_order]
test = test[test_col_order]

print(f"\nTrain columns ({len(train.columns)}): {list(train.columns)}")
print(f"Test columns ({len(test.columns)}):  {list(test.columns)}")

Train columns not in order: set()
Order columns not in train: set()
Test columns not in order:  set()
Order columns not in test:  set()

Train columns (28): ['PassengerId', 'Survived', 'Pclass', 'Sex', 'Age', 'Fare', 'FareLog', 'SibSp', 'Parch', 'FamilySize', 'FamilySizeBin', 'IsAlone', 'HasCabin', 'TicketGroupSize', 'AgeMissing', 'Deck_A', 'Deck_B', 'Deck_C', 'Deck_D', 'Deck_E', 'Deck_F', 'Deck_G', 'Deck_T', 'Embarked_Q', 'Embarked_S', 'Title_Master', 'Title_Miss', 'Title_Mrs']
Test columns (27):  ['PassengerId', 'Pclass', 'Sex', 'Age', 'Fare', 'FareLog', 'SibSp', 'Parch', 'FamilySize', 'FamilySizeBin', 'IsAlone', 'HasCabin', 'TicketGroupSize', 'AgeMissing', 'Deck_A', 'Deck_B', 'Deck_C', 'Deck_D', 'Deck_E', 'Deck_F', 'Deck_G', 'Deck_T', 'Embarked_Q', 'Embarked_S', 'Title_Master', 'Title_Miss', 'Title_Mrs']


## 5. Validation

Verify all features are numeric, no missing values, and schemas are aligned.

In [6]:
# Check all columns are numeric
print("=== Type Check ===")
for name, df in [("Train", train), ("Test", test)]:
    non_numeric = df.select_dtypes(exclude="number").columns.tolist()
    print(f"{name} non-numeric columns: {non_numeric if non_numeric else 'None ✓'}")

# Check for missing values
print("\n=== Missing Values ===")
for name, df in [("Train", train), ("Test", test)]:
    missing = df.isnull().sum()
    missing = missing[missing > 0]
    if len(missing) == 0:
        print(f"{name}: No missing values ✓")
    else:
        print(f"{name} missing:\n{missing}")

# Check schema alignment (test should have all train feature columns)
print("\n=== Schema Alignment ===")
train_features = set(train.columns) - {"PassengerId", "Survived"}
test_features = set(test.columns) - {"PassengerId"}
in_train_not_test = train_features - test_features
in_test_not_train = test_features - train_features
print(f"Features in train not test: {in_train_not_test if in_train_not_test else 'None ✓'}")
print(f"Features in test not train: {in_test_not_train if in_test_not_train else 'None ✓'}")

# Final shapes
print(f"\n=== Final Shapes ===")
print(f"Train: {train.shape} ({train.shape[1] - 2} features + ID + target)")
print(f"Test:  {test.shape} ({test.shape[1] - 1} features + ID)")

=== Type Check ===
Train non-numeric columns: None ✓
Test non-numeric columns: None ✓

=== Missing Values ===
Train: No missing values ✓
Test: No missing values ✓

=== Schema Alignment ===
Features in train not test: None ✓
Features in test not train: None ✓

=== Final Shapes ===
Train: (891, 28) (26 features + ID + target)
Test:  (418, 27) (26 features + ID)


## 6. Train/Test Split Statistics

No separate validation partition is created — per 1.3, evaluation uses stratified 5-fold CV on the 891-row training set during the modeling phase. The Kaggle-provided split is canonical.

In [7]:
# Train target distribution
print("=== Train Target Distribution ===")
surv_counts = train["Survived"].value_counts().sort_index()
surv_pct = train["Survived"].value_counts(normalize=True).sort_index() * 100
for val in surv_counts.index:
    print(f"  Survived={val}: {surv_counts[val]} ({surv_pct[val]:.1f}%)")
print(f"  Class ratio (0:1): {surv_counts[0]/surv_counts[1]:.2f}:1")

# Feature summary statistics
print("\n=== Train Feature Summary ===")
feature_cols = [c for c in train.columns if c not in ("PassengerId", "Survived")]
print(train[feature_cols].describe().round(2).to_string())

# Test feature summary (no target)
print("\n=== Test Feature Summary ===")
test_feature_cols = [c for c in test.columns if c != "PassengerId"]
print(test[test_feature_cols].describe().round(2).to_string())

=== Train Target Distribution ===
  Survived=0: 549 (61.6%)
  Survived=1: 342 (38.4%)
  Class ratio (0:1): 1.61:1

=== Train Feature Summary ===
       Pclass     Sex     Age    Fare  FareLog   SibSp   Parch  FamilySize  FamilySizeBin  IsAlone  HasCabin  TicketGroupSize  AgeMissing  Deck_A  Deck_B  Deck_C  Deck_D  Deck_E  Deck_F  Deck_G  Deck_T  Embarked_Q  Embarked_S  Title_Master  Title_Miss  Title_Mrs
count  891.00  891.00  891.00  891.00   891.00  891.00  891.00      891.00         891.00   891.00    891.00           891.00       891.0  891.00  891.00  891.00  891.00  891.00  891.00  891.00  891.00      891.00      891.00        891.00      891.00     891.00
mean     2.31    0.35   29.37   32.20     2.96    0.52    0.38        1.90           0.47     0.60      0.23             2.12         0.2    0.02    0.05    0.07    0.04    0.04    0.01    0.00    0.00        0.09        0.73          0.04        0.21       0.14
std      0.84    0.48   13.25   49.69     0.97    1.10    0.81    

## 7. Save Formatted Datasets

In [8]:
train_out = PROCESSED_DIR / "train_formatted.csv"
test_out = PROCESSED_DIR / "test_formatted.csv"

train.to_csv(train_out, index=False)
test.to_csv(test_out, index=False)

# Verify round-trip
train_check = pd.read_csv(train_out)
test_check = pd.read_csv(test_out)

assert train_check.shape == train.shape, f"Train shape mismatch: {train_check.shape} vs {train.shape}"
assert test_check.shape == test.shape, f"Test shape mismatch: {test_check.shape} vs {test.shape}"
assert list(train_check.columns) == list(train.columns), "Train column order mismatch"
assert list(test_check.columns) == list(test.columns), "Test column order mismatch"

print(f"✓ Saved {train_out} — {train.shape[0]} rows × {train.shape[1]} cols")
print(f"✓ Saved {test_out} — {test.shape[0]} rows × {test.shape[1]} cols")
print(f"\nRound-trip validation passed ✓")

✓ Saved /Users/tba8ydd/Documents/claude-template/data/processed/train_formatted.csv — 891 rows × 28 cols
✓ Saved /Users/tba8ydd/Documents/claude-template/data/processed/test_formatted.csv — 418 rows × 27 cols

Round-trip validation passed ✓


## 8. Final Schema & Loading Instructions

### Loading the modeling-ready data:

```python
import pandas as pd

train = pd.read_csv("data/processed/train_formatted.csv")
test = pd.read_csv("data/processed/test_formatted.csv")

# Separate features and target
feature_cols = [c for c in train.columns if c not in ("PassengerId", "Survived")]
X_train = train[feature_cols]
y_train = train["Survived"]
X_test = test[feature_cols]
```

### Dataset Card

| Property | Value |
|----------|-------|
| **Name** | titanic-modeling-dataset-v1 |
| **Created** | 2026-03-31 |
| **Created by** | Thibauld |
| **Source data** | Kaggle Titanic (train.csv, test.csv) |
| **Pipeline** | 3.1 Select → 3.2 Clean → 3.3 Construct → 3.5 Format |
| **Records** | 1,309 total (Train: 891, Test: 418) |
| **Features** | 26 (13 numeric + 13 one-hot indicators) |
| **Target** | Survived (0=deceased, 1=survived) |
| **Granularity** | One row per passenger |
| **Known limitations** | Age ~20% imputed; Cabin 77% missing (captured via HasCabin/Deck); small dataset (891 training rows) |

In [9]:
# Final schema
print("=== Final Schema (Train) ===")
print(f"{'#':<3} {'Field':<20} {'Type':<10} {'Role':<10} {'Non-Null':<10}")
print("-" * 55)
for i, col in enumerate(train.columns, 1):
    role = "ID" if col == "PassengerId" else "Target" if col == "Survived" else "Feature"
    dtype = str(train[col].dtype)
    non_null = train[col].notna().sum()
    print(f"{i:<3} {col:<20} {dtype:<10} {role:<10} {non_null}/{len(train)}")

print(f"\nTotal features: {len(train.columns) - 2}")
print(f"Total records: {len(train)} (train) + {len(test)} (test) = {len(train) + len(test)}")

=== Final Schema (Train) ===
#   Field                Type       Role       Non-Null  
-------------------------------------------------------
1   PassengerId          int64      ID         891/891
2   Survived             int64      Target     891/891
3   Pclass               int64      Feature    891/891
4   Sex                  int64      Feature    891/891
5   Age                  float64    Feature    891/891
6   Fare                 float64    Feature    891/891
7   FareLog              float64    Feature    891/891
8   SibSp                int64      Feature    891/891
9   Parch                int64      Feature    891/891
10  FamilySize           int64      Feature    891/891
11  FamilySizeBin        int64      Feature    891/891
12  IsAlone              int64      Feature    891/891
13  HasCabin             int64      Feature    891/891
14  TicketGroupSize      int64      Feature    891/891
15  AgeMissing           int64      Feature    891/891
16  Deck_A               int64  